# MNIST Toy: SFT then RL (winner_only / winner_only_floor / maxRL / RAFT / GRPO)\n

Toy setup:
- Input: downsampled binary MNIST image tokens (values in 0..9, practically 0/1).
- Output tokens length = `n+1`: first token is class `y`, then recurrence `x_i = (a*x_{i-1}+b) % p` with `p=10`.
- SFT until class accuracy >= 60% or `max_sft_steps=1000`.
- RL from same SFT checkpoint for each method: `winner_only`, `winner_only_floor`, `maxrl`, `raft`, `grpo`.\n

Both `n` (recurrence length) and `N` (rollouts per prompt) are configurable below.

In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)

# Core toy parameters
p = 10
n = 6  # recurrence steps after class token
N = 8  # rollouts per prompt in RL
a, b = 3, 1

# Data + model sizes
img_side = 14  # downsampled MNIST side length -> sequence length 196
train_subset = 12000
val_subset = 2000
test_subset = 2000
batch_size = 64

# SFT
max_sft_steps = 1000
sft_target_acc = 0.60
sft_eval_every = 100
sft_lr = 2e-3

# RL
rl_steps = 400
rl_eval_every = 50
rl_lr = 8e-4
methods = ["winner_only", "winner_only_floor", "maxrl", "raft", "grpo"]

print({"n": n, "N": N, "max_sft_steps": max_sft_steps, "rl_steps": rl_steps})

In [ ]:
class TinyCausalTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=10,
        d_model=128,
        nhead=4,
        num_layers=3,
        ff=256,
        max_len=256,
        dropout=0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff,
            dropout=dropout,
            batch_first=True,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        h = self.tok(x) * (self.d_model**0.5) + self.pos(pos)
        # causal mask: True means blocked in PyTorch transformer
        mask = torch.triu(
            torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1
        )
        h = self.enc(h, mask=mask)
        return self.out(h)

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((img_side, img_side)),
        transforms.ToTensor(),
    ]
)

train_full = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_full = datasets.MNIST("./data", train=False, download=True, transform=transform)

rng = np.random.RandomState(0)
train_idx = rng.choice(len(train_full), train_subset + val_subset, replace=False)
train_split = train_idx[:train_subset]
val_split = train_idx[train_subset:]
test_idx = rng.choice(len(test_full), test_subset, replace=False)

train_ds = Subset(train_full, train_split)
val_ds = Subset(train_full, val_split)
test_ds = Subset(test_full, test_idx)

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)

img_len = img_side * img_side
out_len = n + 1
max_len = img_len + out_len


def make_targets(y):
    # y: [B] in 0..9
    B = y.shape[0]
    t = torch.zeros(B, out_len, dtype=torch.long, device=y.device)
    t[:, 0] = y
    for i in range(1, out_len):
        t[:, i] = (a * t[:, i - 1] + b) % p
    return t


def image_to_tokens(x):
    # x: [B,1,H,W] in [0,1] -> binary tokens 0/1 (subset of 0..9)
    return (x[:, 0] > 0.5).long().flatten(1)


def build_sft_batch(x, y):
    img_tok = image_to_tokens(x)
    tgt = make_targets(y)
    seq = torch.cat([img_tok, tgt], dim=1)
    return img_tok, tgt, seq


print("img_len:", img_len, "out_len:", out_len, "max_len:", max_len)

In [ ]:
def class_acc_from_context(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            img_tok = image_to_tokens(x)
            logits = model(img_tok)[:, -1, :]
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return correct / max(total, 1)


def seq_exact_acc_greedy(model, loader):
    model.eval()
    exact = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            img_tok = image_to_tokens(x)
            tgt = make_targets(y)
            cur = img_tok
            outs = []
            for _ in range(out_len):
                logits = model(cur)[:, -1, :]
                nxt = logits.argmax(dim=-1)
                outs.append(nxt)
                cur = torch.cat([cur, nxt.unsqueeze(1)], dim=1)
            gen = torch.stack(outs, dim=1)
            exact += (gen == tgt).all(dim=1).sum().item()
            total += y.numel()
    return exact / max(total, 1)


def sft_train():
    model = TinyCausalTransformer(max_len=max_len).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=sft_lr, weight_decay=1e-2)
    step = 0
    history = []

    train_iter = iter(train_loader)
    while step < max_sft_steps:
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)
        _, _, seq = build_sft_batch(x, y)

        logits = model(seq[:, :-1])
        targets = seq[:, 1:]

        # Loss only on output segment predictions
        # output tokens are at original positions [img_len .. img_len+out_len-1]
        # in shifted targets these are indices [img_len-1 .. img_len+out_len-2]
        mask = torch.zeros_like(targets, dtype=torch.bool)
        mask[:, img_len - 1 :] = True
        ce = F.cross_entropy(
            logits.reshape(-1, p), targets.reshape(-1), reduction="none"
        ).reshape_as(targets)
        loss = ce[mask].mean()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        step += 1
        if step % sft_eval_every == 0 or step == 1:
            val_cls = class_acc_from_context(model, val_loader)
            val_exact = seq_exact_acc_greedy(model, val_loader)
            history.append(
                {
                    "step": step,
                    "sft_loss": float(loss.item()),
                    "val_cls_acc": val_cls,
                    "val_seq_exact": val_exact,
                }
            )
            print(
                f"SFT step={step:4d} loss={loss.item():.4f} val_cls={val_cls:.4f} val_exact={val_exact:.4f}"
            )
            if val_cls >= sft_target_acc:
                print(f"SFT early stop at step {step} (val_cls >= {sft_target_acc})")
                break

    return model, pd.DataFrame(history)


sft_model, sft_hist = sft_train()
display(sft_hist.tail())
print("SFT test cls acc:", class_acc_from_context(sft_model, test_loader))
print("SFT test seq exact:", seq_exact_acc_greedy(sft_model, test_loader))

In [ ]:
@torch.no_grad()
def rollout_stats(model, loader, max_batches=10):
    model.eval()
    rs = []
    ent = []
    seen = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        img_tok = image_to_tokens(x)
        tgt = make_targets(y)
        # B = y.size(0)
        cur = img_tok
        gen = []
        for _ in range(out_len):
            logits = model(cur)[:, -1, :]
            probs = logits.softmax(-1)
            dist = torch.distributions.Categorical(probs=probs)
            tok = dist.sample()
            gen.append(tok)
            ent.append(dist.entropy().mean().item())
            cur = torch.cat([cur, tok.unsqueeze(1)], dim=1)
        gen = torch.stack(gen, dim=1)
        r = (gen == tgt).float().mean(dim=1)
        rs.append(r.mean().item())
        seen += 1
        if seen >= max_batches:
            break
    return float(np.mean(rs)), float(np.mean(ent))


def rl_step_loss(model, img_tok, tgt, method="winner_only", N=8):
    # B = img_tok.size(0)
    rewards = []
    logp_means = []

    for _ in range(N):
        cur = img_tok
        gen = []
        logps = []
        for _ in range(out_len):
            logits = model(cur)[:, -1, :]
            dist = torch.distributions.Categorical(logits=logits)
            tok = dist.sample()
            lp = dist.log_prob(tok)
            gen.append(tok)
            logps.append(lp)
            cur = torch.cat([cur, tok.unsqueeze(1)], dim=1)

        gen = torch.stack(gen, dim=1)
        lp = torch.stack(logps, dim=1).mean(dim=1)  # [B]
        r = (gen == tgt).float().mean(dim=1)  # [B], continuous reward in [0,1]
        rewards.append(r)
        logp_means.append(lp)

    rewards = torch.stack(rewards, dim=0)  # [N,B]
    logp_means = torch.stack(logp_means, dim=0)  # [N,B]

    r_mean = rewards.mean(dim=0, keepdim=True)
    r_std = rewards.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)

    if method == "winner_only":
        raw = torch.where(rewards > r_mean, 1.0 - r_mean, torch.zeros_like(rewards))
        denom = (raw > 0).float().sum(dim=0, keepdim=True).clamp_min(1.0)
        w = raw / denom
    elif method == "maxrl":
        raw = torch.clamp(rewards - r_mean, min=0.0)
        denom = (raw > 0).float().sum(dim=0, keepdim=True).clamp_min(1.0)
        w = raw / denom
    elif method == "raft":
        raw = torch.clamp(rewards, min=0.0)
        denom = (raw > 0).float().sum(dim=0, keepdim=True).clamp_min(1.0)
        w = raw / denom
    elif method == "grpo":
        w = (rewards - r_mean) / r_std / float(N)
    else:
        raise ValueError(method)

    loss = -(w * logp_means).sum(dim=0).mean()
    avg_reward = rewards.mean().detach().item()
    return loss, avg_reward


def run_rl(method):
    model = copy.deepcopy(sft_model).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=rl_lr, weight_decay=1e-2)

    hist = []
    train_iter = iter(train_loader)
    for step in range(1, rl_steps + 1):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)
        img_tok = image_to_tokens(x)
        tgt = make_targets(y)

        loss, train_reward = rl_step_loss(model, img_tok, tgt, method=method, N=N)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % rl_eval_every == 0 or step == 1:
            cls = class_acc_from_context(model, val_loader)
            seq_exact = seq_exact_acc_greedy(model, val_loader)
            val_reward, ent = rollout_stats(model, val_loader, max_batches=8)
            row = {
                "method": method,
                "step": step,
                "loss": float(loss.item()),
                "train_reward": float(train_reward),
                "val_reward": val_reward,
                "val_cls_acc": cls,
                "val_seq_exact": seq_exact,
                "val_entropy": ent,
            }
            hist.append(row)
            print(
                f"[{method}] step={step:4d} loss={row['loss']:.4f} train_r={train_reward:.4f} val_r={val_reward:.4f} val_cls={cls:.4f}"
            )

    return model, pd.DataFrame(hist)

In [ ]:
all_hist = []
trained = {}
for m in methods:
    model_m, hist_m = run_rl(m)
    trained[m] = model_m
    all_hist.append(hist_m)

df = pd.concat(all_hist, ignore_index=True)
display(df.head())

# Final test metrics
rows = []
for m, model_m in trained.items():
    rows.append(
        {
            "method": m,
            "test_cls_acc": class_acc_from_context(model_m, test_loader),
            "test_seq_exact": seq_exact_acc_greedy(model_m, test_loader),
            "test_rollout_reward": rollout_stats(model_m, test_loader, max_batches=12)[
                0
            ],
        }
    )
summary = pd.DataFrame(rows).sort_values("test_rollout_reward", ascending=False)
display(summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
metrics = [
    ("val_reward", "Validation rollout reward"),
    ("val_cls_acc", "Validation class accuracy"),
    ("val_seq_exact", "Validation sequence exact-match"),
    ("val_entropy", "Validation policy entropy"),
]

for (metric, title), ax in zip(metrics, axes.flatten()):
    for m in methods:
        g = df[df["method"] == m].sort_values("step")
        ax.plot(g["step"], g[metric], marker="o", label=m)
    ax.set_title(title)
    ax.set_xlabel("RL step")
    ax.grid(alpha=0.25)

axes[0, 0].legend()
plt.tight_layout()
plt.show()

# SFT curve
if len(sft_hist) > 0:
    fig = plt.figure(figsize=(8, 4))
    plt.plot(
        sft_hist["step"], sft_hist["val_cls_acc"], marker="o", label="SFT val cls acc"
    )
    plt.plot(
        sft_hist["step"],
        sft_hist["val_seq_exact"],
        marker="o",
        label="SFT val seq exact",
    )
    plt.axhline(
        sft_target_acc, color="gray", linestyle="--", label="target 60% class acc"
    )
    plt.xlabel("SFT step")
    plt.ylabel("Metric")
    plt.title("SFT progress")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()